In [ ]:
import os
import pickle
import torch
import numpy as np
import logging
import torchvision
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transform
from DataSet import CustomStrmDataset
from model import *
from tqdm import tqdm
from datetime import datetime
from utils import *
import tractogramReader as tr
from matplotlib import pyplot as plt


In [4]:
RANDOM_SEED = 42
device = set_device()

GPU is enabled. 



In [14]:
start_time = datetime.now()
current_time = start_time
root_train = '../../Synthetic_tck/256res/20M_N256.tck'
list_train_IDs = np.arange(20000000)
dataset_train = CustomStrmDataset(root_train, 
                            list_train_IDs,
                            numberOfStreamlines=len(list_train_IDs),
                            transformData=True, 
                            res=256,
                            device=device,
                            seed=RANDOM_SEED)

'''root_test = '../../Synthetic_tck/256res/20M_N256.tck'
list_test_IDs = np.arange(20000000)
dataset_test = CustomStrmDataset(root_test, 
                            list_test_IDs,
                            numberOfStreamlines=len(list_test_IDs),
                            transformData=True, 
                            res=256,
                            device=device,
                            seed=RANDOM_SEED)'''


"root_test = '../../Synthetic_tck/256res/20M_N256.tck'\nlist_test_IDs = np.arange(20000000)\ndataset_test = CustomStrmDataset(root_test, \n                            list_test_IDs,\n                            numberOfStreamlines=len(list_test_IDs),\n                            transformData=True, \n                            res=256,\n                            device=device,\n                            seed=RANDOM_SEED)"

In [16]:
res = 256
data_shape = [3, res]
latent_dim = 64
n_kernels = 32
kernel_size = 4
learning_rate = 1e-3
batch_size = 256
num_workers = 1
momentum = 0.9

#lr=6.68e-4, weight_decay=0.13

In [17]:
seed_all(seed=RANDOM_SEED)
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

train_loader = torch.utils.data.DataLoader(dataset_train, 
                                           batch_size=batch_size,                                         
                                           shuffle=True, 
                                           num_workers=num_workers,
                                           worker_init_fn=seed_worker,
                                           generator=g)
'''test_loader = torch.utils.data.DataLoader(dataset_test, 
                                           batch_size=batch_size,                                         
                                           shuffle=True, 
                                           num_workers=num_workers,
                                           worker_init_fn=seed_worker,
                                           generator=g)'''

Using Seed Number 42


'test_loader = torch.utils.data.DataLoader(dataset_test, \n                                           batch_size=batch_size,                                         \n                                           shuffle=True, \n                                           num_workers=num_workers,\n                                           worker_init_fn=seed_worker,\n                                           generator=g)'

In [ ]:
# Define model and optimizer
model = ConvAE_256res.ConvAutoEncoder(x_dim=data_shape, latent_dim=latent_dim, n_kernels=n_kernels, kernel_size=kernel_size)
model = model.to(device)
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate)


NameError: name 'ConvAutoEncoder' is not defined

In [11]:
# log and save
num_run = int(1)
while os.path.isdir('train_'+str(num_run))==True:
    num_run += 1
num_run = str(num_run)
os.system('mkdir train_'+num_run)
os.system('mkdir train_'+num_run+'/checkpoints')
log = open('train_'+num_run+'/log.txt', 'w')
params = open('train_'+num_run+'/params.txt', 'w')
params.write('latent_dimension\t'+str(latent_dim)+'\n')
params.write('numb_kernels\t'+str(n_kernels)+'\n')
params.write('resolution\t'+str(res)+'\n')
params.write('learn rate\t'+str(learning_rate)+'\n')
params.write('size_kernel\t'+str(kernel_size)+'\n')
params.close()
log.write('Epoch\t loss'+'\n')
log.close()

In [12]:
def eval_model(model, data_loader, num_batches=np.inf, device='cpu'):
    """
    Evaluate model
    Args:
    model: nn.Module derived class
      The model which is to be evaluated
    data_loader: Iterable
    num_batches: Integer
    device: String
    Returns:
    mean of log loss
    """
    model.eval()
    loss_log = []

    with torch.no_grad():
        for batch_id, strm_coor_batch in enumerate(data_loader):
            
            # If we only evaluate a number of batches, stop after we reach that number
            if batch_id > num_batches:
                break
            strm_coor_batch = strm_coor_batch.to(device)
            strm_coor_batch_recon = model(strm_coor_batch)
            # Evaluate model and loss on minibatch
            loss_log.append(F.l1_loss(strm_coor_batch_recon, strm_coor_batch).item())

    return np.mean(loss_log)

In [ ]:
# Run training
LOG_FREQ = 400
VAL_FREQ = 4000
MAX_EPOCHS = 50
start_time = datetime.now()
current_time = start_time

metrics = {'train_loss':[],
           'val_loss':[],
           'val_idx':[]}

step_idx = 0
for epoch in tqdm(range(MAX_EPOCHS)): 

    running_loss = 0.
    epoch_loss = []
    model.train()
    seed_all(seed=RANDOM_SEED) # the place is correct?
    for batch_id, strm_coor_batch in enumerate(train_loader):
        step_idx += 1
        
        strm_coor_batch = strm_coor_batch.to(device)
        optimizer.zero_grad()
        # autoencoder reconstruction
        strm_coor_batch_recon = model(strm_coor_batch)
        torch.autograd.set_detect_anomaly(True)

        #loss = streamDist(strm_coor_batch_recon, strm_coor_batch)
        loss = F.l1_loss(strm_coor_batch_recon, strm_coor_batch)
        
        # backpropagation
        loss.backward()
        #for name, param in model.named_parameters():
        #    print(name, torch.isfinite(param.grad).all())
        # one step of the optmizer 
        optimizer.step()
        
        metrics['train_loss'].append(loss.cpu().item())
        epoch_loss.append(loss.cpu().item())

        if batch_id % VAL_FREQ == (VAL_FREQ - 1):
            # Get an estimate of the validation accuracy with 100 batches
            val_loss = eval_model (model, 
                                   train_loader,
                                   num_batches=100,
                                   device=device)
            metrics['val_idx'].append(step_idx)
            metrics['val_loss'].append(val_loss)

            print(f"[VALID] Epoch {epoch + 1} - Batch {batch_id + 1} - " f"Loss: {val_loss:.3f}")

        running_loss += loss.cpu().item()
        # Print every LOG_FREQ minibatches
        if batch_id % LOG_FREQ == (LOG_FREQ-1):
            print(f"[TRAIN] Epoch {epoch + 1} - Batch {batch_id + 1} - " f"Loss: {running_loss / LOG_FREQ:.3f}")
            running_loss = 0.
            
    # Saving 
    dur_last = datetime.now() - current_time
    current_time = datetime.now()
    if (epoch) % 2 == 0:
        log_tr_loss = open('train_'+num_run+'/log.txt', 'a')
        log_tr_loss.write(str(epoch)+'\t training loss='+str(np.mean(epoch_loss))+'\n')

        print('[{}/{}]'.format(epoch,MAX_EPOCHS)+' training '+str(num_run))
        print("Current time", current_time)
        print("Duration total: {}".format(current_time-start_time))
        print("Duration last iteration: {}".format(dur_last))

    if (epoch) % 5 == 0: 
        torch.save(model.state_dict(),'train_'+num_run+'/checkpoints/AE'+str(epoch)+'.pth')
        
        
with open('train_'+num_run+'/log_tr_loss.pickle', 'wb') as file:
    pickle.dump(metrics, file)